# 🎯 Açık Kaynak Modeller — VARYANT Test Notebook (FINAL)

Bu notebook, açık kaynak modelleri **258 soruluk Varyant veri seti** üzerinde test eder.

**Base testinden farkları:**
1. Prompt'a JSON-only kuralı eklendi (Llama'nın markdown sarma alışkanlığı engellendi)
2. max_new_tokens 768 → 1024 (Llama'nın truncation hataları azaltıldı)
3. extract_json_from_response'a 4 aşamalı fallback parser eklendi (JSON parse hataları otomatik kurtarılır)
4. Dataset mode kilitli: "varyant"

**Test edilecek modeller (sırasıyla):**
1. Qwen 2.5 7B Instruct (Bölüm C)
2. Llama 3.1 8B Instruct (Bölüm D)
3. Gemma 2 9B IT (Bölüm E, opsiyonel)

---

## 📋 Kullanım Sırası

1. **Bölüm A** (Setup) - Her runtime başında zorunlu
2. **Bölüm B** (Yardımcı fonksiyonlar + iyileştirmeler) - Her runtime başında zorunlu
3. Hangi model için test yapacaksan o bölümü (C/D/E) çalıştır
4. Her model bitince GPU temizlenir, sonraki modele geçilebilir

---
## 🔧 BÖLÜM A — Setup
---

In [ ]:
# ============================================================
# A.1 — Kütüphaneleri kur
# ============================================================

!pip install -q transformers accelerate bitsandbytes sentencepiece
!pip install -q sentence-transformers
!pip install -q tqdm scipy

print("✓ Kütüphaneler kuruldu")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 36.1 MB/s eta 0:00:00
object address  : 0x7ab42d74eec0
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr
^C
✓ Kütüphaneler kuruldu


In [ ]:
# Sorunlu paketleri kaldır ve uyumlu versiyonları kur
!pip uninstall -y transformers torch -q
!pip install -q torch==2.5.1
!pip install -q transformers==4.46.0
!pip install -q accelerate bitsandbytes sentencepiece
!pip install -q sentence-transformers
!pip install -q tqdm scipy

print("\n⚠️ ŞIMDI ÖNEMLI: Runtime → Restart session yap!")
print("   Sonra A.2'den itibaren sırayla devam et")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 118.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 112.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# ============================================================
# A.2 — Drive bağla, path'i hazırla, cache temizle
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import sys, os, shutil, gc, json, time

PROJECT_FOLDER = "/content/drive/MyDrive/BitirmeProjesi"
sys.path.insert(0, PROJECT_FOLDER)

# Eski cache'i temizle
pycache_path = f"{PROJECT_FOLDER}/core/__pycache__"
if os.path.exists(pycache_path):
    shutil.rmtree(pycache_path)
    print("✓ __pycache__ temizlendi")

utils_path = f"{PROJECT_FOLDER}/core/faithfulness_utils.py"
assert os.path.exists(utils_path), f"faithfulness_utils.py bulunamadı: {utils_path}"
print(f"✓ faithfulness_utils.py bulundu")

from core.faithfulness_utils import *
print("✓ faithfulness_utils fonksiyonları import edildi")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ __pycache__ temizlendi
✓ faithfulness_utils.py bulundu
✓ faithfulness_utils fonksiyonları import edildi


In [ ]:
# ============================================================
# A.3 — Transformers, GPU kontrolü, HuggingFace token
# ============================================================

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from sentence_transformers import SentenceTransformer
from huggingface_hub import login
from google.colab import userdata

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name} ({vram:.1f} GB VRAM)")
else:
    raise RuntimeError("GPU yok! Runtime → Change runtime type → GPU seç")

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    login(token=HF_TOKEN)
    print("✓ HuggingFace login başarılı")
except Exception as e:
    print(f"⚠ HF token sorunu: {e}")

RuntimeError: Failed to import transformers.pipelines because of the following error (look up to see its traceback):
operator torchvision::nms does not exist

In [ ]:
# ============================================================
# A.4 — Veri seti: VARYANT (sabit)
# ============================================================

DATASET_MODE = "varyant"

import pandas as pd
from datetime import datetime

CHECKPOINT_PATH = f"{PROJECT_FOLDER}/checkpoints/"
RESULTS_PATH = f"{PROJECT_FOLDER}/results/"
os.makedirs(CHECKPOINT_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

# Varyant CSV dosyasının adı (gerekirse değiştir)
DATA_PATH = f"{PROJECT_FOLDER}/Sorular e-tablo - Variants (1).csv"
DATASET_TAG = "varyant"
EXPECTED_ROWS = 258

# Eğer yukarıdaki ad bulunamazsa, BitirmeProjesi klasöründeki varyant dosyasını otomatik ara
if not os.path.exists(DATA_PATH):
    print(f"⚠ Tahmin edilen yol bulunamadı: {DATA_PATH}")
    print("  BitirmeProjesi klasöründe 'varyant' içeren CSV aranıyor...")
    candidates = [
        f for f in os.listdir(PROJECT_FOLDER)
        if f.endswith('.csv') and 'varyant' in f.lower()
    ]
    if candidates:
        DATA_PATH = f"{PROJECT_FOLDER}/{candidates[0]}"
        print(f"  ✓ Bulundu: {DATA_PATH}")
    else:
        all_csv = [f for f in os.listdir(PROJECT_FOLDER) if f.endswith('.csv')]
        print(f"  ✗ Varyant CSV bulunamadı. Klasördeki tüm CSV'ler:")
        for f in all_csv:
            print(f"     - {f}")
        raise FileNotFoundError("Varyant CSV bulunamadı. DATA_PATH'i elle ayarla.")

# Config'i utils'e set et
set_config(
    vector_w=0.4,
    nli_w=0.6,
    llm_model="local-open-source",
    temp=0,
    max_tok=2048,
    checkpoint_path=CHECKPOINT_PATH
)

TODAY = datetime.now().strftime('%Y%m%d')

df = load_and_preprocess_data(DATA_PATH)

print(f"\n✓ Veri seti yüklendi: {len(df)} satır")
print(f"  Mode: {DATASET_MODE} (beklenen: {EXPECTED_ROWS})")
print(f"  Dosya: {DATA_PATH.split('/')[-1]}")
print(f"  Tarih: {TODAY}")

In [ ]:
df = df.rename(columns={'variant_id': 'question_id'})
print(f"✓ Kolonlar: {df.columns.tolist()}")

✓ Kolonlar: ['question_id', 'base_question_id', 'category', 'difficulty', 'variant_type', 'morphological_feature', 'context', 'question', 'gold_evidence', 'gold_answer', 'gold_evidence_list', 'evidence_count']


In [ ]:
# A.5 ALTERNATİF — Embedding/NLI'yi CPU'da yükle
EMBEDDING_MODEL = "intfloat/multilingual-e5-large"
NLI_MODEL = "joeddav/xlm-roberta-large-xnli"

print("► Embedding model CPU'da yükleniyor...")
emb_model = SentenceTransformer(EMBEDDING_MODEL, device='cpu')
print(f"  ✓ {EMBEDDING_MODEL}")

print("► NLI pipeline CPU'da yükleniyor...")
nli_pipe = pipeline(
    "zero-shot-classification",
    model=NLI_MODEL,
    device=-1   # CPU
)
print(f"  ✓ {NLI_MODEL}")

set_models(emb_model, nli_pipe, None, None, None)

print("\n✓ Skorlama altyapısı hazır (CPU)")

► Embedding model CPU'da yükleniyor...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✓ intfloat/multilingual-e5-large
► NLI pipeline CPU'da yükleniyor...


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✓ joeddav/xlm-roberta-large-xnli

✓ Skorlama altyapısı hazır (CPU)


In [ ]:
# ============================================================
# A.5 — Embedding ve NLI modellerini yükle
# ============================================================

EMBEDDING_MODEL = "intfloat/multilingual-e5-large"
NLI_MODEL = "joeddav/xlm-roberta-large-xnli"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("► Embedding model yükleniyor...")
emb_model = SentenceTransformer(EMBEDDING_MODEL, device=device)
print(f"  ✓ {EMBEDDING_MODEL}")

print("► NLI pipeline yükleniyor (zero-shot-classification)...")
nli_pipe = pipeline(
    "zero-shot-classification",
    model=NLI_MODEL,
    device=0 if device == "cuda" else -1
)
print(f"  ✓ {NLI_MODEL}")

set_models(emb_model, nli_pipe, None, None, None)

test_sem = calculate_semantic_similarity("merhaba dünya", "selam dünya")
test_nli = calculate_nli_score("Bugün hava güzel", "Hava bugün güzel")
print(f"\n  Sanity check:")
print(f"    semantic_similarity test = {test_sem:.4f}")
print(f"    nli_score test = {test_nli['nli_score']:.4f}")
print("\n✓ Skorlama altyapısı hazır")

► Embedding model yükleniyor...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

  ✓ joeddav/xlm-roberta-large-xnli

  Sanity check:
    semantic_similarity test = 0.9570
    nli_score test = 0.3886

✓ Skorlama altyapısı hazır


---
## 🔧 BÖLÜM B — İyileştirilmiş Yardımcı Fonksiyonlar

Base testindeki sorunları gideren 3 iyileştirme entegre edildi:
1. Prompt'a JSON-only kuralı
2. max_new_tokens 1024
3. 4 aşamalı fallback JSON parser
---

In [ ]:
# ============================================================
# B.1 — İyileştirilmiş prompt (JSON-only kuralı eklendi)
# ============================================================

OPEN_SOURCE_SYSTEM_PROMPT = SYSTEM_PROMPT + """

ÇOK ÖNEMLİ FORMAT KURALI:
- Cevabını YALNIZCA tek bir JSON nesnesi olarak ver.
- JSON'u markdown kod bloğu (```json) içinde verme.
- JSON'dan önce veya sonra HİÇBİR açıklama, selamlama, özet ekleme.
- 'reasoning' alanını TEK bir string olarak yaz, ayrı string listesi olarak değil.
- 'reasoning' içinde satır geçişi kullanmak istiyorsan \\n yaz, literal newline değil.
- Cevabını JSON'un kapanış süslü parantezi ile bitir.
"""

def build_user_prompt(context: str, question: str) -> str:
    return f"""BAĞLAM:
{context}

SORU:
{question}

Yukarıdaki kurallara uygun, SADECE geçerli JSON formatında ve markdown sarmadan cevap ver."""

print("✓ İyileştirilmiş prompt hazır")
print(f"  SYSTEM_PROMPT uzunluğu: {len(OPEN_SOURCE_SYSTEM_PROMPT)} karakter")

✓ İyileştirilmiş prompt hazır
  SYSTEM_PROMPT uzunluğu: 1083 karakter


In [ ]:
# ============================================================
# B.2 — Model yükleme ve generation (max_new_tokens=1024)
# ============================================================

def load_model_4bit(model_id: str):
    """Modeli 4-bit quantization ile yükler (VRAM tasarrufu)."""
    print(f"► Model yükleniyor: {model_id}")
    print("  (İlk indirme uzun sürebilir, sonraki çalıştırmalar cache'den hızlı)")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"✓ Model yüklendi: {model_id}")
    return model, tokenizer


def _supports_system_role(tokenizer) -> bool:
    """Tokenizer'ın chat template'i system role'unü destekliyor mu test et."""
    try:
        tokenizer.apply_chat_template(
            [{"role": "system", "content": "test"}, {"role": "user", "content": "test"}],
            tokenize=False,
            add_generation_prompt=True,
        )
        return True
    except Exception:
        return False


def generate_response(model, tokenizer, context: str, question: str, max_new_tokens: int = 1024) -> str:
    """İyileştirilmiş generation: max_new_tokens=1024 (önceki 768)."""
    user_prompt = build_user_prompt(context, question)

    if _supports_system_role(tokenizer):
        messages = [
            {"role": "system", "content": OPEN_SOURCE_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]
    else:
        combined = f"{OPEN_SOURCE_SYSTEM_PROMPT}\n\n{user_prompt}"
        messages = [{"role": "user", "content": combined}]

    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True)
    return response


def cleanup_model(model, tokenizer):
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print("✓ GPU belleği temizlendi")


print("✓ Model fonksiyonları tanımlandı (max_new_tokens=1024)")

✓ Model fonksiyonları tanımlandı (max_new_tokens=1024)


In [ ]:
# ============================================================
# B.3 — İyileştirilmiş JSON parser (4 aşamalı fallback)
# ============================================================
# faithfulness_utils.py'deki extract_json_from_response'u override eder.
# Orijinal başarısız olursa 4 fallback stratejisi sırayla denenir.

import re as _re
import json as _json

_original_extract_json = extract_json_from_response


def _fb_markdown(raw):
    """```json ... ``` veya ``` ... ``` bloğunu temizle."""
    m = _re.search(r'```(?:json)?\s*(\{.*?\})\s*```', raw, _re.DOTALL)
    if m:
        try:
            return _json.loads(m.group(1))
        except _json.JSONDecodeError:
            return None
    return None


def _fb_literal_newline(raw):
    """String içindeki literal \\n/\\r/\\t karakterlerini escape eder."""
    s = raw.strip()
    if not s.startswith('{'):
        idx = s.find('{')
        if idx == -1:
            return None
        s = s[idx:]

    depth = 0
    in_string = False
    escape = False
    end_idx = -1
    for i, ch in enumerate(s):
        if escape:
            escape = False
            continue
        if ch == '\\':
            escape = True
            continue
        if ch == '"':
            in_string = not in_string
            continue
        if in_string:
            continue
        if ch == '{':
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0:
                end_idx = i
                break

    if end_idx == -1:
        return None

    candidate = s[:end_idx+1]
    result_chars = []
    in_string = False
    escape = False
    for ch in candidate:
        if escape:
            result_chars.append(ch)
            escape = False
            continue
        if ch == '\\':
            result_chars.append(ch)
            escape = True
            continue
        if ch == '"':
            in_string = not in_string
            result_chars.append(ch)
            continue
        if in_string and ch in '\n\r\t':
            if ch == '\n':
                result_chars.append('\\n')
            elif ch == '\r':
                result_chars.append('\\r')
            elif ch == '\t':
                result_chars.append('\\t')
        else:
            result_chars.append(ch)

    fixed = ''.join(result_chars)
    try:
        return _json.loads(fixed)
    except _json.JSONDecodeError:
        return None


def _fb_concatenated(raw):
    """reasoning ayrı string'lere bölünmüşse birleştir."""
    s = raw.strip()
    if not s.startswith('{'):
        idx = s.find('{')
        if idx == -1:
            return None
        s = s[idx:]

    reasoning_match = _re.search(r'"reasoning"\s*:\s*"([^"]*)"', s, _re.DOTALL)
    answer_match = _re.search(
        r'"answer"\s*:\s*(?:"([^"]*)"|([-+]?\d+(?:\.\d+)?)|(true|false|null))',
        s
    )
    confidence_match = _re.search(r'"confidence"\s*:\s*(\d+(?:\.\d+)?)', s)

    if reasoning_match is None:
        return None

    reasoning = reasoning_match.group(1)
    end_of_reasoning = reasoning_match.end()
    if answer_match:
        start_of_answer = answer_match.start()
        middle = s[end_of_reasoning:start_of_answer]
    else:
        middle = s[end_of_reasoning:]

    orphan_strings = _re.findall(r'"([^"]*)"', middle)
    extra = ' '.join(orphan_strings)
    if extra:
        reasoning = f"{reasoning} {extra}".strip()

    if answer_match:
        ans = answer_match.group(1) or answer_match.group(2) or answer_match.group(3)
    else:
        ans = None

    if confidence_match:
        try:
            conf = float(confidence_match.group(1))
        except ValueError:
            conf = None
    else:
        conf = None

    if ans is None:
        return None

    return {
        'reasoning': reasoning,
        'answer': ans,
        'confidence': conf if conf is not None else 50,
    }


def _fb_regex(raw):
    """Son çare: regex ile reasoning/answer/confidence çek."""
    reasoning_match = _re.search(
        r'"reasoning"\s*:\s*"((?:[^"\\]|\\.)*?)"',
        raw, _re.DOTALL
    )
    answer_match = _re.search(
        r'"answer"\s*:\s*(?:"((?:[^"\\]|\\.)*?)"|([-+]?\d+(?:\.\d+)?))',
        raw, _re.DOTALL
    )
    confidence_match = _re.search(r'"confidence"\s*:\s*(\d+(?:\.\d+)?)', raw)

    if reasoning_match is None or answer_match is None:
        return None

    reasoning = reasoning_match.group(1)
    ans = answer_match.group(1) or answer_match.group(2)
    conf = float(confidence_match.group(1)) if confidence_match else 50

    return {
        'reasoning': reasoning,
        'answer': ans,
        'confidence': conf,
    }


def extract_json_from_response(raw):
    """
    İyileştirilmiş parser. Önce orijinal denenir, başarısız olursa 4 fallback uygulanır.
    """
    if not raw or not isinstance(raw, str):
        return None

    try:
        parsed = _original_extract_json(raw)
        if parsed is not None and isinstance(parsed, dict):
            if 'reasoning' in parsed and 'answer' in parsed:
                return parsed
    except Exception:
        pass

    for strategy in (_fb_markdown, _fb_literal_newline, _fb_concatenated, _fb_regex):
        try:
            parsed = strategy(raw)
            if parsed is not None and isinstance(parsed, dict):
                if 'reasoning' in parsed and 'answer' in parsed:
                    return parsed
        except Exception:
            continue

    return None


# Sanity check
_t1 = extract_json_from_response('```json\n{"reasoning": "test", "answer": "42", "confidence": 80}\n```')
_t2 = extract_json_from_response('{\n"reasoning": "Adım 1.\nAdım 2.",\n"answer": 12.5,\n"confidence": 100\n}')
print("✓ İyileştirilmiş extract_json_from_response aktif")
print(f"  Markdown test: {'✓' if _t1 and _t1.get('answer') == '42' else '✗'}")
print(f"  Literal newline test: {'✓' if _t2 and _t2.get('answer') == 12.5 else '✗'}")

✓ İyileştirilmiş extract_json_from_response aktif
  Markdown test: ✓
  Literal newline test: ✓


In [ ]:
# ============================================================
# B.4 — Generation: checkpoint'li tüm veri seti üzerinde model çalıştır
# ============================================================

from tqdm.notebook import tqdm

def run_generation(model, tokenizer, df, model_short_name: str, dataset_tag: str):
    """Modeli tüm veri seti üzerinde çalıştırır, checkpoint kaydeder."""
    checkpoint_file = f"{CHECKPOINT_PATH}generation_{model_short_name}_{dataset_tag}_{TODAY}.json"

    results = []
    processed_ids = set()
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, 'r', encoding='utf-8') as f:
            results = json.load(f)
        processed_ids = {r['question_id'] for r in results}
        print(f"✓ Checkpoint bulundu: {len(processed_ids)} soru zaten işlenmiş")

    to_process = df[~df['question_id'].isin(processed_ids)]

    if len(to_process) == 0:
        print("✓ Tüm sorular zaten işlenmiş — checkpoint'ten devam ediliyor")
    else:
        print(f"► {len(to_process)} soru işlenecek (model: {model_short_name})")

        for idx, row in tqdm(to_process.iterrows(), total=len(to_process), desc=f"Gen [{model_short_name}]"):
            context = str(row['context'])
            question = str(row['question'])

            try:
                raw = generate_response(model, tokenizer, context, question)
                parsed = extract_json_from_response(raw)

                if parsed is None or not isinstance(parsed, dict):
                    results.append({
                        'question_id': row['question_id'],
                        'model_raw_response': raw,
                        'model_reasoning': None,
                        'model_answer': None,
                        'model_confidence': None,
                        'error': 'JSON_PARSE_ERROR',
                    })
                else:
                    results.append({
                        'question_id': row['question_id'],
                        'model_raw_response': raw,
                        'model_reasoning': parsed.get('reasoning'),
                        'model_answer': parsed.get('answer'),
                        'model_confidence': parsed.get('confidence'),
                        'error': None,
                    })
            except Exception as e:
                results.append({
                    'question_id': row['question_id'],
                    'model_raw_response': '',
                    'model_reasoning': None,
                    'model_answer': None,
                    'model_confidence': None,
                    'error': str(e),
                })

            if len(results) % 5 == 0:
                with open(checkpoint_file, 'w', encoding='utf-8') as f:
                    json.dump(results, f, ensure_ascii=False, indent=2)

        with open(checkpoint_file, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"✓ Checkpoint kaydedildi: {checkpoint_file}")

    results_df = pd.DataFrame(results)
    merged = df.merge(results_df, on='question_id', how='left')

    success_count = merged['model_answer'].notna().sum()
    error_count = merged['error'].notna().sum()
    print(f"\n✓ Generation özeti:")
    print(f"   Başarılı: {success_count}/{len(merged)}")
    if error_count > 0:
        print(f"   Hatalı (JSON parse vb.): {error_count}")

    return merged

print("✓ run_generation tanımlandı")

✓ run_generation tanımlandı


In [ ]:
# ============================================================
# B.5 — Skorlama ve özet fonksiyonları
# ============================================================

def run_scoring(merged_df: pd.DataFrame, model_short_name: str, dataset_tag: str) -> pd.DataFrame:
    valid = merged_df[
        merged_df['error'].isna() &
        merged_df['model_reasoning'].notna() &
        (merged_df['model_reasoning'].astype(str).str.len() > 0)
    ].copy().reset_index(drop=True)

    print(f"► Skorlanacak satır sayısı: {len(valid)} / {len(merged_df)}")
    if len(valid) == 0:
        print("⚠ Hiç geçerli satır yok")
        return merged_df

    df_evaluated = run_evaluation(valid)

    print("\n► Soft accuracy hesaplanıyor...")
    soft_rows = []
    for _, row in tqdm(df_evaluated.iterrows(), total=len(df_evaluated), desc="Soft Acc"):
        try:
            r = calculate_soft_accuracy(row['model_answer'], row['gold_answer'])
        except Exception as e:
            r = {'soft_match': False, 'accuracy_score': 0, 'match_type': f'error: {e}',
                 'semantic_sim': 0.0, 'nli_entailment': 0.0}
        soft_rows.append({
            'question_id': row['question_id'],
            'soft_match': r['soft_match'],
            'soft_accuracy_score': r['accuracy_score'],
            'match_type': r['match_type'],
            'answer_semantic_sim': r['semantic_sim'],
            'answer_nli_entailment': r['nli_entailment'],
        })
    soft_df = pd.DataFrame(soft_rows)
    df_evaluated = df_evaluated.merge(soft_df, on='question_id', how='left')

    output_path = f"{RESULTS_PATH}evaluation_acik_{model_short_name}_{dataset_tag}_{TODAY}.csv"
    df_evaluated.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n✓ Skorlama tamamlandı: {output_path}")

    return df_evaluated


def print_summary(df_evaluated: pd.DataFrame, model_display_name: str):
    print("\n" + "=" * 70)
    print(f"📊 ÖZET — {model_display_name}")
    print("=" * 70)
    print(f"Toplam soru: {len(df_evaluated)}")

    if 'accuracy_score' in df_evaluated.columns:
        exact_rate = df_evaluated['accuracy_score'].mean() * 100
        print(f"\nExact Accuracy: {exact_rate:.1f}%  ({df_evaluated['accuracy_score'].sum()}/{len(df_evaluated)})")

    if 'soft_accuracy_score' in df_evaluated.columns:
        soft_rate = df_evaluated['soft_accuracy_score'].mean() * 100
        print(f"Soft Accuracy:  {soft_rate:.1f}%  ({df_evaluated['soft_accuracy_score'].sum()}/{len(df_evaluated)})")

    if 'faithfulness_score' in df_evaluated.columns:
        print(f"\nOrtalama semantic_similarity: {df_evaluated['semantic_similarity'].mean():.3f}")
        print(f"Ortalama nli_score:           {df_evaluated['nli_score'].mean():.3f}")
        print(f"Ortalama faithfulness_score:  {df_evaluated['faithfulness_score'].mean():.3f}")
        print(f"Ortalama evidence_recall:     {df_evaluated['evidence_recall'].mean():.3f}")

    if 'confidence_normalized' in df_evaluated.columns:
        print(f"\nOrtalama confidence: {df_evaluated['confidence_normalized'].mean()*100:.1f}%")
        print(f"Ortalama epistemic_gap: {df_evaluated['epistemic_gap'].mean():.3f}")

        print("\nHallucination risk dağılımı:")
        for risk, cnt in df_evaluated['hallucination_risk'].value_counts().items():
            print(f"   {risk}: {cnt} ({cnt/len(df_evaluated)*100:.1f}%)")

        print("\nCalibration tipi dağılımı:")
        for cal, cnt in df_evaluated['calibration_type'].value_counts().items():
            print(f"   {cal}: {cnt} ({cnt/len(df_evaluated)*100:.1f}%)")

    print("=" * 70)


print("✓ run_scoring ve print_summary tanımlandı")

✓ run_scoring ve print_summary tanımlandı


---
## 🔧 BÖLÜM C — Qwen 2.5 7B Instruct (VARYANT)

**Tahmini süre:** ~45 dk - 1 saat (258 soru)
**Beklenen başarı oranı:** %98+ (base'de %98 idi)
---

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
MODEL_SHORT_NAME = "qwen25_7b"
MODEL_DISPLAY_NAME = "Qwen 2.5 7B Instruct (varyant)"

model, tokenizer = load_model_4bit(MODEL_ID)
merged = run_generation(model, tokenizer, df, MODEL_SHORT_NAME, DATASET_TAG)
cleanup_model(model, tokenizer)

df_evaluated = run_scoring(merged, MODEL_SHORT_NAME, DATASET_TAG)
print_summary(df_evaluated, MODEL_DISPLAY_NAME)

NameError: name 'load_model_4bit' is not defined

---
## 🔧 BÖLÜM D — Llama 3.1 8B Instruct (VARYANT)

**Tahmini süre:** ~1.5 saat (258 soru, max_new_tokens=1024)
**Beklenen başarı oranı:** %95+ (base'de iyileştirme öncesi %62, iyileştirme sonrası kurtarmayla %93 → şimdi direkt yüksek)
---

In [ ]:
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_SHORT_NAME = "llama31_8b"
MODEL_DISPLAY_NAME = "Llama 3.1 8B Instruct (varyant)"

model, tokenizer = load_model_4bit(MODEL_ID)
merged = run_generation(model, tokenizer, df, MODEL_SHORT_NAME, DATASET_TAG)
cleanup_model(model, tokenizer)

df_evaluated = run_scoring(merged, MODEL_SHORT_NAME, DATASET_TAG)
print_summary(df_evaluated, MODEL_DISPLAY_NAME)

---
## 🔧 BÖLÜM E — Gemma 2 9B IT (VARYANT) — Opsiyonel

Zaman kalırsa çalıştır.
**Tahmini süre:** ~1.5 saat
---

In [ ]:
MODEL_ID = "google/gemma-2-9b-it"
MODEL_SHORT_NAME = "gemma2_9b"
MODEL_DISPLAY_NAME = "Gemma 2 9B IT (varyant)"

model, tokenizer = load_model_4bit(MODEL_ID)
merged = run_generation(model, tokenizer, df, MODEL_SHORT_NAME, DATASET_TAG)
cleanup_model(model, tokenizer)

df_evaluated = run_scoring(merged, MODEL_SHORT_NAME, DATASET_TAG)
print_summary(df_evaluated, MODEL_DISPLAY_NAME)

---
## 🔧 BÖLÜM F — Sonradan Skorlama (Opsiyonel)

Generation tamamlandıktan sonra runtime kapanırsa, bu bölüm checkpoint'ten okuyup sadece skorlama yapar. Model yüklemez. Önce Bölüm A ve B'yi çalıştırmış olman gerekir.
---

In [ ]:
# Bozuk dosyayı sil
import os
scoring_checkpoint = f"{CHECKPOINT_PATH}scoring_qwen25_7b_varyant_{TODAY}.json"
if os.path.exists(scoring_checkpoint):
    os.remove(scoring_checkpoint)
    print(f"✓ Silindi: {scoring_checkpoint}")

✓ Silindi: /content/drive/MyDrive/BitirmeProjesi/checkpoints/scoring_qwen25_7b_varyant_20260515.json


In [ ]:
import json
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

RESCORE_MODEL_SHORT = "qwen25_7b"
RESCORE_DISPLAY = "Qwen 2.5 7B Instruct (varyant)"

# numpy tiplerini Python native'ye çevir
def to_jsonable(obj):
    """numpy/pandas tiplerini standart Python tipine çevir."""
    if isinstance(obj, (np.floating, np.float32, np.float64)):
        return float(obj)
    if isinstance(obj, (np.integer, np.int32, np.int64)):
        return int(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, dict):
        return {k: to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(x) for x in obj]
    return obj


def score_one_row(row):
    result = {'question_id': row['question_id']}
    try:
        evidence_list = row['gold_evidence_list']
        reasoning = str(row['model_reasoning'])

        faith_result = calculate_faithfulness(reasoning, evidence_list)
        result['semantic_similarity'] = float(faith_result['semantic_similarity'])
        result['nli_score'] = float(faith_result['nli_score'])
        result['faithfulness_score'] = float(faith_result['faithfulness_score'])
        result['evidence_recall'] = float(faith_result['evidence_recall'])

        model_ans = str(row['model_answer']).strip().lower()
        gold_ans = str(row['gold_answer']).strip().lower()
        result['accuracy_score'] = 1 if model_ans == gold_ans else 0

        try:
            conf_raw = float(row['model_confidence']) if row['model_confidence'] is not None else 50
        except (ValueError, TypeError):
            conf_raw = 50
        result['confidence_normalized'] = conf_raw / 100

        ep_metrics = calculate_epistemic_metrics(
            conf_raw,
            result['faithfulness_score'],
            result['accuracy_score']
        )
        # ep_metrics içindeki numpy tiplerini temizle
        for k, v in ep_metrics.items():
            result[k] = to_jsonable(v)

        soft = calculate_soft_accuracy(row['model_answer'], row['gold_answer'])
        result['soft_match'] = bool(soft['soft_match'])
        result['soft_accuracy_score'] = int(soft['accuracy_score'])
        result['match_type'] = str(soft['match_type'])
        result['answer_semantic_sim'] = float(soft['semantic_sim'])
        result['answer_nli_entailment'] = float(soft['nli_entailment'])

        result['error'] = None
    except Exception as e:
        result['error'] = str(e)

    return result


# Generation checkpoint'i yükle
import glob
candidates = glob.glob(f"{CHECKPOINT_PATH}generation_{RESCORE_MODEL_SHORT}_{DATASET_TAG}_*.json")
gen_checkpoint = sorted(candidates)[-1]
print(f"✓ Generation checkpoint: {gen_checkpoint}")

with open(gen_checkpoint, 'r', encoding='utf-8') as f:
    gen_results = json.load(f)
gen_df = pd.DataFrame(gen_results)
merged = df.merge(gen_df, on='question_id', how='left')

valid = merged[
    merged['error'].isna() &
    merged['model_reasoning'].notna() &
    (merged['model_reasoning'].astype(str).str.len() > 0)
].copy().reset_index(drop=True)
print(f"► Skorlanacak satır: {len(valid)} / {len(merged)}")

# Eski skorlama checkpoint varsa yükle
scoring_checkpoint = f"{CHECKPOINT_PATH}scoring_{RESCORE_MODEL_SHORT}_{DATASET_TAG}_{TODAY}.json"
scored_dict = {}
if os.path.exists(scoring_checkpoint):
    try:
        with open(scoring_checkpoint, 'r', encoding='utf-8') as f:
            scored_list = json.load(f)
        scored_dict = {r['question_id']: r for r in scored_list}
        print(f"✓ Skorlama checkpoint bulundu: {len(scored_dict)} zaten skorlanmış")
    except json.JSONDecodeError:
        print("⚠ Eski skorlama checkpoint bozuk, atlanıyor")
        scored_dict = {}

to_score = valid[~valid['question_id'].isin(scored_dict.keys())]
print(f"► İşlenecek: {len(to_score)} soru\n")

for _, row in tqdm(to_score.iterrows(), total=len(to_score), desc="Skorlama"):
    res = score_one_row(row)
    scored_dict[res['question_id']] = res

    if len(scored_dict) % 10 == 0:
        with open(scoring_checkpoint, 'w', encoding='utf-8') as f:
            json.dump(list(scored_dict.values()), f, ensure_ascii=False, indent=2)

# Son kayıt
with open(scoring_checkpoint, 'w', encoding='utf-8') as f:
    json.dump(list(scored_dict.values()), f, ensure_ascii=False, indent=2)
print(f"\n✓ Skorlama checkpoint kaydedildi: {scoring_checkpoint}")

# Final CSV
scored_df = pd.DataFrame(list(scored_dict.values()))
final_df = valid.merge(scored_df, on='question_id', how='left')

output_path = f"{RESULTS_PATH}evaluation_acik_{RESCORE_MODEL_SHORT}_{DATASET_TAG}_{TODAY}.csv"
final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"✓ CSV kaydedildi: {output_path}")

print_summary(final_df, RESCORE_DISPLAY)

✓ Generation checkpoint: /content/drive/MyDrive/BitirmeProjesi/checkpoints/generation_qwen25_7b_varyant_20260515.json
► Skorlanacak satır: 254 / 257
✓ Skorlama checkpoint bulundu: 254 zaten skorlanmış
► İşlenecek: 0 soru



Skorlama: 0it [00:00, ?it/s]


✓ Skorlama checkpoint kaydedildi: /content/drive/MyDrive/BitirmeProjesi/checkpoints/scoring_qwen25_7b_varyant_20260515.json
✓ CSV kaydedildi: /content/drive/MyDrive/BitirmeProjesi/results/evaluation_acik_qwen25_7b_varyant_20260515.csv

📊 ÖZET — Qwen 2.5 7B Instruct (varyant)
Toplam soru: 254

Exact Accuracy: 18.9%  (48/254)
Soft Accuracy:  27.4%  (68.0/254)

Ortalama semantic_similarity: 0.872
Ortalama nli_score:           0.501
Ortalama faithfulness_score:  0.649
Ortalama evidence_recall:     1.000

Ortalama confidence: 95.6%
Ortalama epistemic_gap: 0.307

Hallucination risk dağılımı:
   ORTA: 154 (60.6%)
   DÜŞÜK: 81 (31.9%)
   YÜKSEK: 19 (7.5%)

Calibration tipi dağılımı:
   Overconfident: 225 (88.6%)
   İyi Kalibre: 29 (11.4%)


In [ ]:
# A.3 ALTERNATİF — CPU için
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from sentence_transformers import SentenceTransformer
from huggingface_hub import login
from google.colab import userdata

print(f"CUDA available: {torch.cuda.is_available()}")
print("CPU modunda çalışıyoruz, GPU gerekmiyor")

# HF token (Llama için bile gerekmiyor şu an, ama yine de yükle)
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    login(token=HF_TOKEN)
    print("✓ HuggingFace login başarılı")
except Exception as e:
    print(f"⚠ HF token sorunu: {e}")

KeyboardInterrupt: 